# Candy & Tea Order Predictor

A time series regression model used to predict the ordering frequency for candy

Future goal: modifying the data to also predict tea orders as well

## Imports
The packages needed to run the model

In [23]:
# Imports
print('This is a test')

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
#NOTE: This is for testing cross validation later... deal with this later
from sklearn.model_selection import cross_val_score, KFold




This is a test


## Data Prep

In [24]:
# Train & Validation Data Setup

file_path = 'data/candy_data.csv'
df = pd.read_csv('data/candy_data.csv')
print(df.head())

# creating the structure for the model
# I want to filter for whatever item I assess (ex: candy, tea, etc.)
df['item'] = df['item'].str.lower().str.strip()
df['is_candy'] = df['item'].str.contains(r'\bcandy\b', regex=True, na=False)

# additional data cleaning
df['cost'] = df['cost'].str.replace('USD', ' ').astype(float)
df['req_date'] = pd.to_datetime(df['req_date'])

# defining filtered candy dataframe
candy_df = df[df['is_candy']]
#NOTE: this code converts the req_date to a datetime object for time series purposes 
candy_df['req_date'] = pd.to_datetime(candy_df['req_date'])
candy_df = candy_df.set_index('req_date')
print(candy_df.index)
#NOTE: we are going to test weekly vs monthly date counts to see which gives a better regression
weekly = candy_df.resample('W').size().reset_index(name='order_count')
monthly = candy_df.resample('ME').size().reset_index(name='order_count')

# Train test split for candy data
TrainData, TestData = train_test_split(candy_df, test_size=0.2, shuffle=False, random_state=42)
print(f'Train Data Size: {len(TrainData)}')
print(f'Test Data Size: {len(TestData)}')

print(candy_df.head())

    item            req_id   req_date requester    status budget_status  \
0  candy  NWUNV/REQ2302418  4/28/2026   QDR0260  Complete         Valid   
1    tea  NWUNV/REQ2297481  4/15/2026   QDR0260  Complete         Valid   
2  candy  NWUNV/REQ2296641  4/13/2026   QDR0260  Complete         Valid   
3    tea  NWUNV/REQ2293614   4/4/2026   QDR0260  Complete         Valid   
4  candy  NWUNV/REQ2292744   4/1/2026   QDR0260  Complete         Valid   

         cost  
0  184.17 USD  
1   63.08 USD  
2  420.01 USD  
3   63.08 USD  
4  205.25 USD  
DatetimeIndex(['2026-04-28', '2026-04-13', '2026-04-01', '2026-03-02',
               '2026-02-19', '2026-02-09', '2026-01-24', '2026-01-15',
               '2026-01-06', '2025-10-21', '2026-07-16', '2026-06-04',
               '2026-05-18', '2026-05-01', '2026-04-28', '2025-10-01',
               '2025-09-24', '2025-08-21', '2025-08-05', '2025-07-15',
               '2025-06-16', '2025-05-22', '2025-05-15', '2025-05-12',
               '2025-05-06'